# PFE ML — Period-Fix With Diagnostics And Training Fixes

This notebook builds on `period_fix_rebuild.ipynb`. It does **not** redo the clean layer or the feature build — both are already on Drive from the previous run. Instead, it:

1. Pulls the latest `data-extraction` branch (which now ships two training-pipeline fixes — see below).
2. Verifies the period-fix is still in place (`company_identity` manifest reports `schema_version: 2` and a period grain).
3. Runs the temporal-validity check (activity_code variation across years) on the **full** feature table, not the smoke sample.
4. Runs a feature-quality audit (null rate, variance, pairwise correlation) and writes a CSV you can attach to the report.
5. Trains run 8 with the same 2M-row cap as runs 1, 2, and 7 so the comparison isolates the effect of the training fixes.
6. Displays the new run alongside the seven prior entries in `model_run_comparison.csv` / `.png`.

## Training fixes shipped on the branch

Both fixes live in `app/tools/train_continuity_model.py` and are picked up automatically by `git pull`.

- **Missing-indicator imputer.** `SimpleImputer(strategy="median", add_indicator=True)` adds a binary column per numeric column with NaNs, so the classifier can distinguish a genuine zero from "no financial filing on record". With ~43% of financial cells missing, median-only imputation collapsed those two signals.
- **Four dropped columns.** `EXCLUDE_COLUMNS` now also excludes `has_confidential_financials` (identical coefficient to `has_financial_data` in run 7 → perfect collinearity) and the three `formalities_count_*` columns (coefficient 0.0 → zero variance, source table empty).

## 1. Runtime

**High-RAM CPU.** No GPU/TPU. The training step is ~5 minutes once features exist; the audit cell takes ~30 seconds on a 200k-row sample.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DATA_LAKE = f'{DRIVE_ROOT}/data-lake'
DUCKDB_TMP = '/content/pfein_duckdb_tmp'

START_YEAR = 2017
END_YEAR = 2024
TRAIN_MAX_ROWS = 2_000_000
TARGET = 'continuity_risk_12m_label'
AUDIT_SAMPLE_ROWS = 200_000

FEATURES_GLOB = f'{DATA_LAKE}/features/company_year_features/**/*.parquet'
LABELS_GLOB = f'{DATA_LAKE}/features/risk_labels/**/*.parquet'

Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)

print('BACKEND_DIR  =', BACKEND_DIR)
print('DRIVE_ROOT   =', DRIVE_ROOT)
print('BRANCH       =', BRANCH)
print('YEARS        =', START_YEAR, '-', END_YEAR)
print('TRAIN_CAP    =', TRAIN_MAX_ROWS)

## 2. Pull Code And Install Dependencies

Make sure you've committed and pushed the two training-pipeline fixes to `data-extraction` before running this — otherwise the pull below brings in the old script and the run becomes identical to run 7.

In [ ]:
import os

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!pip install -q -r collabs/requirements-colab.txt

In [ ]:
import re

train_script_text = (Path(BACKEND_DIR) / 'app' / 'tools' / 'train_continuity_model.py').read_text(encoding='utf-8')
has_indicator = 'add_indicator=True' in train_script_text
has_exclude = 'has_confidential_financials' in train_script_text.split('def main')[0]
print(f'Pipeline fixes present in pulled script:')
print(f'  add_indicator=True in imputer       : {has_indicator}')
print(f'  has_confidential_financials excluded: {has_exclude}')
if not (has_indicator and has_exclude):
    raise SystemExit('Training-pipeline fixes are missing from the pulled branch. Commit them on data-extraction and push before re-running this cell.')

## 3. Verify The Clean Layer Is Still Period-Fixed

Read the existing manifest from Drive. We do **not** rebuild clean — the previous notebook already produced `schema_version: 2` and a period grain. If this assertion fails, re-run `period_fix_rebuild.ipynb` end-to-end first.

In [ ]:
import json

manifest_path = Path(f'{DATA_LAKE}/clean/company_identity/_manifest.json')
if not manifest_path.exists():
    raise SystemExit(
        f'Missing manifest at {manifest_path}. '
        'Run period_fix_rebuild.ipynb first to rebuild the clean layer.'
    )
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print(json.dumps(
    {k: manifest.get(k) for k in ('schema_version', 'grain', 'row_count', 'generated_at')},
    indent=2,
))

assert manifest.get('schema_version') == 2, (
    f"company_identity schema_version expected 2, got {manifest.get('schema_version')!r}"
)
assert 'period' in manifest.get('grain', ''), (
    f"company_identity grain expected period-aware, got {manifest.get('grain')!r}"
)
print('\nClean layer OK — period grain is in place.')

## 4. Verify The Feature Table Exists

Check that `company_year_features` and `risk_labels` parquet files are on Drive from the previous full build. If either is missing, the next cell raises and you should re-run the period-fix rebuild notebook.

In [ ]:
import duckdb

features_root = Path(f'{DATA_LAKE}/features/company_year_features')
labels_root = Path(f'{DATA_LAKE}/features/risk_labels')

feature_files = list(features_root.rglob('*.parquet')) if features_root.exists() else []
label_files = list(labels_root.rglob('*.parquet')) if labels_root.exists() else []
if not feature_files:
    raise SystemExit(f'No feature parquet under {features_root}. Re-run the period-fix rebuild notebook.')
if not label_files:
    raise SystemExit(f'No label parquet under {labels_root}. Re-run the period-fix rebuild notebook.')
print(f'Feature parquet files: {len(feature_files)}')
print(f'Label parquet files:   {len(label_files)}')

con = duckdb.connect()
counts = con.execute(f'''
    SELECT
        (SELECT count(*) FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)) AS feature_rows,
        (SELECT count(*) FROM read_parquet('{LABELS_GLOB}',   union_by_name=true)) AS label_rows,
        (SELECT min(prediction_year) FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)) AS min_year,
        (SELECT max(prediction_year) FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)) AS max_year
''').df()
print('\nFeature/label counts:')
print(counts.to_string(index=False))
con.close()

## 5. Temporal-Validity Check On The Full Feature Table

The previous notebook ran this check on a 50k-company smoke sample and reported 225 SIRENs with activity-code variation across years. Now run it against the **full** feature parquet so the number in the thesis report reflects the real population, not a sample.

In [ ]:
con = duckdb.connect()
for column in ('activity_code', 'legal_category_code', 'employee_size_bracket', 'administrative_status_at_cutoff'):
    validity = con.execute(f'''
        WITH per_siren AS (
            SELECT siren, count(DISTINCT {column}) AS distinct_codes
            FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)
            WHERE {column} IS NOT NULL
            GROUP BY siren
        )
        SELECT
            count(*) AS total_sirens_with_value,
            sum(CASE WHEN distinct_codes > 1 THEN 1 ELSE 0 END) AS sirens_with_changed_value,
            ROUND(100.0 * sum(CASE WHEN distinct_codes > 1 THEN 1 ELSE 0 END) / count(*), 3) AS pct_changed
        FROM per_siren
    ''').df()
    print(f'\n{column}:')
    print(validity.to_string(index=False))
con.close()

## 6. Feature Quality Audit

Samples ~200k rows from the feature table and computes, per column: null rate, standard deviation (numeric), and distinct count. Then lists numeric pairs with `|corr| >= 0.95` to expose the kind of duplication that gave `has_confidential_financials` and `has_financial_data` identical coefficients last run.

The CSV `feature_quality_report.csv` is written to the ml-artifacts root so it can be attached to the thesis report.

In [ ]:
import pandas as pd

con = duckdb.connect()
audit_sample = con.execute(f'''
    SELECT *
    FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)
    USING SAMPLE {AUDIT_SAMPLE_ROWS} ROWS
''').df()
con.close()
print(f'Audit sample shape: {audit_sample.shape}')

ID_COLS = {'siren', 'prediction_date', 'prediction_year', 'company_name'}
audit_cols = [c for c in audit_sample.columns if c not in ID_COLS]

rows = []
for col in audit_cols:
    series = audit_sample[col]
    null_rate = float(series.isna().mean())
    if pd.api.types.is_bool_dtype(series):
        numeric = series.astype('Int64').astype(float)
        dtype = 'boolean'
        std = float(numeric.std(skipna=True)) if numeric.notna().any() else 0.0
        distinct = int(numeric.nunique(dropna=True))
    elif pd.api.types.is_numeric_dtype(series):
        dtype = 'numeric'
        std = float(series.std(skipna=True)) if series.notna().any() else 0.0
        distinct = int(series.nunique(dropna=True))
    else:
        dtype = 'categorical'
        std = None
        distinct = int(series.nunique(dropna=True))
    rows.append({
        'column': col, 'dtype': dtype, 'null_rate': null_rate,
        'std': std, 'distinct': distinct,
    })

stats_df = pd.DataFrame(rows).sort_values('null_rate', ascending=False)
print('\nPer-column quality (top 20 by null rate):')
print(stats_df.head(20).to_string(index=False))

zero_var = stats_df[(stats_df['dtype'].isin(['numeric', 'boolean'])) & (stats_df['std'].fillna(0) == 0)]
print('\nZero-variance numeric/boolean columns (carry no information):')
if zero_var.empty:
    print('  (none)')
else:
    print(zero_var[['column', 'null_rate', 'distinct']].to_string(index=False))

audit_csv = Path(DRIVE_ROOT) / 'ml-artifacts' / 'feature_quality_report.csv'
audit_csv.parent.mkdir(parents=True, exist_ok=True)
stats_df.to_csv(audit_csv, index=False)
print(f'\nSaved per-column report to: {audit_csv}')

In [ ]:
numeric_audit_cols = [
    col for col in audit_cols
    if pd.api.types.is_numeric_dtype(audit_sample[col]) or pd.api.types.is_bool_dtype(audit_sample[col])
]
X_audit = audit_sample[numeric_audit_cols].copy()
for col in X_audit.columns:
    if pd.api.types.is_bool_dtype(X_audit[col]):
        X_audit[col] = X_audit[col].astype('Int64')
X_audit = X_audit.astype(float)

corr = X_audit.corr().abs()
pairs = []
for i, a in enumerate(numeric_audit_cols):
    for b in numeric_audit_cols[i+1:]:
        value = corr.loc[a, b]
        if pd.notna(value) and value >= 0.95:
            pairs.append((a, b, float(value)))
pairs.sort(key=lambda x: -x[2])
print('Numeric feature pairs with |corr| >= 0.95 (top 20):')
if not pairs:
    print('  (none)')
for a, b, c in pairs[:20]:
    print(f'  {a:42s}  {b:42s}  corr={c:.4f}')

corr_csv = Path(DRIVE_ROOT) / 'ml-artifacts' / 'feature_correlations_high.csv'
pd.DataFrame(pairs, columns=['column_a', 'column_b', 'abs_corr']).to_csv(corr_csv, index=False)
print(f'\nSaved high-correlation pairs to: {corr_csv}')

## 7. Train Run 8 (Period-Valid Features + Training Fixes)

Same 2M-row cap as runs 1, 2, and 7. The only differences from run 7 are the two fixes in `train_continuity_model.py` (missing-indicator imputer + 4 dropped degenerate columns). Anything else changing between run 7 and run 8 in the metrics is attributable to these two fixes.

In [ ]:
import shlex, subprocess, sys

train_cmd = [
    sys.executable, '-u',
    '-m', 'app.tools.train_continuity_model',
    '--data-lake-dir', f'{DRIVE_ROOT}/data-lake',
    '--artifacts-dir', f'{DRIVE_ROOT}/ml-artifacts',
    '--target', TARGET,
    '--train-start-year', str(START_YEAR),
    '--train-end-year', str(END_YEAR),
    '--max-rows', str(TRAIN_MAX_ROWS),
    '--min-rows', '1000',
]
print(' '.join(shlex.quote(p) for p in train_cmd))
subprocess.run(train_cmd, check=True)

metadata_path = Path(DRIVE_ROOT) / 'ml-artifacts' / 'model_metadata.json'
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
print('\n' + json.dumps({
    'run_name': metadata.get('run_name'),
    'model_version': metadata.get('model_version'),
    'rows': metadata.get('rows'),
    'feature_count': metadata.get('feature_count'),
    'excluded_columns_present': metadata.get('excluded_columns_present'),
    'split_strategy': metadata.get('split_strategy'),
    'test_class_counts': metadata.get('test_class_counts'),
    'train_missing_values': metadata.get('train_missing_values'),
    'metrics': {
        k: metadata.get('metrics', {}).get(k)
        for k in ('accuracy', 'roc_auc', 'average_precision',
                  'precision_at_0_5', 'recall_at_0_5', 'f1_at_0_5')
    },
    'run_artifacts_dir': metadata.get('run_artifacts_dir'),
}, indent=2))

## 8. Display Run Artifacts

In [ ]:
from IPython.display import Image, Markdown, display

run_dir = Path(metadata['run_artifacts_dir'])
print('Run folder:')
print(run_dir)

print('\nFiles:')
for path in sorted(run_dir.iterdir()):
    print(' ', path.name)

summary_path = run_dir / 'run_summary.md'
if summary_path.exists():
    display(Markdown(summary_path.read_text(encoding='utf-8')))

image_names = [
    'class_counts_by_year.png',
    'precision_recall_curve.png',
    'roc_curve.png',
    'confusion_matrix_at_0_5.png',
    'score_distribution_by_class.png',
    'threshold_tradeoff.png',
    'top_feature_coefficients.png',
]
for image_name in image_names:
    image_path = run_dir / image_name
    if image_path.exists():
        print('\n' + image_name)
        display(Image(filename=str(image_path)))

comparison_image = Path(DRIVE_ROOT) / 'ml-artifacts' / 'model_run_comparison.png'
if comparison_image.exists():
    print('\nmodel_run_comparison.png')
    display(Image(filename=str(comparison_image)))

## 9. Run 7 vs Run 8 Coefficient Comparison

Side-by-side of the top non-sector features per run so you can show in the report that:

- `has_confidential_financials` is gone (excluded).
- The three `formalities_count_*` columns are gone (excluded).
- Financial features (`latest_revenue`, `latest_net_result`, ...) hopefully move out of the floor now that missing-indicator features absorb the "no filing" signal.

In [ ]:
runs_dir = Path(DRIVE_ROOT) / 'ml-artifacts' / 'runs'
run_folders = sorted(runs_dir.iterdir(), key=lambda p: p.name)
if len(run_folders) < 2:
    print('Need at least two runs for comparison.')
else:
    prev_dir = run_folders[-2]
    new_dir = run_folders[-1]
    print(f'Previous: {prev_dir.name}')
    print(f'New:      {new_dir.name}')

    def top_financial_and_nonsector(path, n=15):
        df = pd.read_csv(path)
        df['is_sector'] = df['feature'].str.startswith(('activity_code_', 'legal_category_code_'))
        return df[~df['is_sector']].head(n)[['feature', 'coefficient']]

    prev_csv = prev_dir / 'feature_coefficients.csv'
    new_csv = new_dir / 'feature_coefficients.csv'
    if prev_csv.exists() and new_csv.exists():
        prev_top = top_financial_and_nonsector(prev_csv).rename(columns={'feature': 'prev_feature', 'coefficient': 'prev_coef'})
        new_top = top_financial_and_nonsector(new_csv).rename(columns={'feature': 'new_feature', 'coefficient': 'new_coef'})
        side_by_side = pd.concat([prev_top.reset_index(drop=True), new_top.reset_index(drop=True)], axis=1)
        print('\nTop non-sector features (sector dummies removed for clarity):')
        print(side_by_side.to_string(index=False))
    else:
        print('feature_coefficients.csv missing from one of the runs.')

In [ ]:
comparison_csv = Path(DRIVE_ROOT) / 'ml-artifacts' / 'model_run_comparison.csv'
if comparison_csv.exists():
    runs_summary = pd.read_csv(comparison_csv)
    keep = [
        'run_name', 'rows', 'feature_count', 'test_rows', 'test_positive_rate',
        'average_precision', 'roc_auc', 'precision_at_0_5', 'recall_at_0_5', 'f1_at_0_5',
    ]
    print('All runs (chronological):')
    print(runs_summary[keep].to_string(index=False))
else:
    print('No model_run_comparison.csv yet.')

## 10. What To Send After The Run

- The newest folder under `ml-artifacts/runs/`: `run_summary.md`, `feature_coefficients.csv`, `precision_recall_curve.png`, `threshold_tradeoff.png`, `top_feature_coefficients.png`.
- `ml-artifacts/feature_quality_report.csv` (null rate / variance / distinct per column).
- `ml-artifacts/feature_correlations_high.csv` (numeric pairs with |corr| ≥ 0.95).
- `ml-artifacts/model_run_comparison.png` and `.csv` (now showing 8 runs).
- The output of section 5 (temporal-validity counts on full data) — those numbers are the headline for the period-fix section of the thesis.
- The output of section 9 (top non-sector coefficients before vs after fixes) — that's the evidence the financial features are now contributing.